# T22: CLIP Single-Stream C++ Demo

This tutorial explains how to build and run a C++ CLIP application with camera or video input. Text queries are encoded with ONNX Runtime, images are encoded asynchronously with a DEEPX NPU, and a Qt GUI displays the similarity scores.

![CLIP single-stream demo](assets/clip-single-sc.png)

## 1. Learning goals

By the end of this tutorial, you will understand:

- how CLIP compares image and text embeddings,
- how the C++ application combines ONNX Runtime and DXRT,
- how camera and video frames are prepared for the image encoder,
- how asynchronous NPU inference and frame skipping work, and
- how to build and run the application.

In [ ]:
from pathlib import Path
import shutil
import subprocess

tutorial_name = "T22-demo-clip-single"
candidates = [
    Path.cwd(),
    Path.cwd() / "notebooks" / tutorial_name,
    Path.cwd().parent,
]
TUTORIAL_ROOT = next(
    (path.resolve() for path in candidates if (path / "app" / "main.cpp").is_file()),
    None,
)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError(f"Could not locate {tutorial_name}")

APP_DIR = TUTORIAL_ROOT / "app"
ASSETS_DIR = TUTORIAL_ROOT / "assets"
print(f"Tutorial root: {TUTORIAL_ROOT}")

## 2. Project layout

```text
T22-demo-clip-single/
├── README.md
├── clip_single.ipynb
├── get_resources.sh
├── assets/
│   ├── models/
│   ├── videos/
│   └── images/
└── app/
    ├── CMakeLists.txt
    ├── build.sh
    ├── run_camera.sh
    ├── run_video.sh
    ├── main.cpp
    ├── clip_tokenizer.cpp
    └── clip_tokenizer.hpp
```

In [ ]:
required_files = [
    TUTORIAL_ROOT / "get_resources.sh",
    APP_DIR / "CMakeLists.txt",
    APP_DIR / "build.sh",
    APP_DIR / "run_camera.sh",
    APP_DIR / "run_video.sh",
    APP_DIR / "main.cpp",
    APP_DIR / "clip_tokenizer.cpp",
    APP_DIR / "clip_tokenizer.hpp",
]
for path in required_files:
    status = "OK" if path.exists() else "MISSING"
    print(f"[{status:7}] {path.relative_to(TUTORIAL_ROOT)}")

## 3. Required resources

`get_resources.sh` is currently an empty placeholder. Before running the demo, place the image encoder, text encoder, BPE vocabulary, and video at the paths below. The ONNX `.data` file is required when the text encoder stores weights as external data.

In [ ]:
resources = [
    ASSETS_DIR / "models" / "ViT-L-14-quickgelu-dfn2b.dxnn",
    ASSETS_DIR / "models" / "ViT-L-14-quickgelu-dfn2b-text.onnx",
    ASSETS_DIR / "models" / "ViT-L-14-quickgelu-dfn2b-text.onnx.data",
    ASSETS_DIR / "models" / "bpe_simple_vocab_16e6.txt.gz",
    ASSETS_DIR / "videos" / "CLIP-demo.mp4",
]
for path in resources:
    status = "READY" if path.is_file() else "MISSING"
    print(f"[{status:7}] {path.relative_to(TUTORIAL_ROOT)}")

## 4. Inference pipeline

```text
Text queries -> BPE tokens -> ONNX Runtime text encoder -> text embeddings
                                                              |
Camera/video -> resize and center crop -> DXRT image encoder -> image embedding
                                                              |
                                                              v
                                      L2 normalization -> dot products -> ranked GUI scores
```

Text embeddings are calculated at startup and cached. Each selected video frame is converted to a 224 x 224 CHW float tensor. DXRT submits image inference asynchronously, so capture and GUI updates do not wait for each NPU request to finish.

In [ ]:
def show_source(relative_path, marker, line_count=50):
    path = APP_DIR / relative_path
    lines = path.read_text(encoding="utf-8").splitlines()
    start = next((index for index, line in enumerate(lines) if marker in line), None)
    if start is None:
        raise ValueError(f"Marker not found: {marker}")
    end = min(len(lines), start + line_count)
    for number in range(start, end):
        print(f"{number + 1:4}: {lines[number]}")

### Command-line options

`AppOptions` defines the model paths, input source, camera settings, frame-skipping interval, and GUI flags. If `--input` is omitted, the application uses the camera selected by `--camera`.

In [ ]:
show_source("main.cpp", "struct AppOptions", line_count=145)

### Image preprocessing

The image is resized while preserving its aspect ratio, center-cropped to 224 x 224, converted from BGR to RGB order, normalized, and stored in CHW layout.

In [ ]:
show_source("main.cpp", "std::vector<float> preprocessFrame", line_count=55)

### Text encoding and caching

`TextEncoder` tokenizes each query and runs the ONNX text encoder. `TextFeatureStore` saves normalized embeddings under `.cache/text_features_cpp/`, keyed by the model and text list.

In [ ]:
show_source("main.cpp", "class TextEncoder", line_count=75)
print("\n--- Text feature cache ---")
show_source("main.cpp", "class TextFeatureStore", line_count=100)

### Asynchronous image encoding

`ImageEncoderAsync` creates a DXRT inference engine with multiple buffers. Each submitted frame owns its input memory until the completion callback copies the 768-value image embedding.

In [ ]:
show_source("main.cpp", "class ImageEncoderAsync", line_count=145)

### Similarity ranking

After optional L2 normalization, the application calculates one dot product per text query. For normalized embeddings, this is cosine similarity. The GUI highlights the strongest matches above the configured score threshold.

In [ ]:
show_source("main.cpp", "void updateSimilarities", line_count=75)

## 5. Inspect the DXNN image encoder

When the model is available, `dxparse` shows the model input and output tensors. The application expects a float input shaped `[1, 3, 224, 224]` and a 768-value image embedding.

In [ ]:
image_encoder = ASSETS_DIR / "models" / "ViT-L-14-quickgelu-dfn2b.dxnn"
if not image_encoder.is_file():
    print(f"Model is not available yet: {image_encoder}")
elif shutil.which("dxparse") is None:
    print("dxparse is not installed or is not on PATH.")
else:
    subprocess.run(["dxparse", "-m", str(image_encoder), "-v"], check=True)

## 6. Build the application

`build.sh` configures CMake in Release mode and runs `make` with all CPU cores reported by `nproc`. Use `./build.sh --clean` to remove the existing build directory first. Building does not require the model or video files.

In [ ]:
subprocess.run(["./build.sh"], cwd=APP_DIR, check=True)

In [ ]:
subprocess.run(["./build/clip_single", "--help"], cwd=APP_DIR, check=True)

## 7. Run the camera demo

The default source is `/dev/video0` at a requested 1920 x 1080 and 30 FPS. Set `RUN_CAMERA = True` after the resources and camera are ready. The GUI blocks the cell until it is closed.

In [ ]:
RUN_CAMERA = False
if RUN_CAMERA:
    subprocess.run(
        ["./run_camera.sh", "--camera", "/dev/video0",
         "--width", "1280", "--height", "720", "--fps", "30"],
        cwd=APP_DIR,
        check=True,
    )
else:
    print("Set RUN_CAMERA = True to start the camera GUI.")

## 8. Run the video demo

`run_video.sh` uses `assets/videos/CLIP-demo.mp4` and loops at the end of the file. Set `RUN_VIDEO = True` after the resources are ready.

In [ ]:
RUN_VIDEO = False
if RUN_VIDEO:
    subprocess.run(["./run_video.sh"], cwd=APP_DIR, check=True)
else:
    print("Set RUN_VIDEO = True to start the video GUI.")

## 9. Useful options

- `--skip-frames 0` runs image inference for every frame.
- `--skip-frames 2` runs image inference for every third frame and is the default.
- `--no-normalize` disables L2 normalization before similarity calculation.
- `--camera SOURCE` selects a camera when `--input` is omitted.
- `--input SOURCE` accepts a camera index, camera device, or video path.
- `--full-screen` opens the Qt window in fullscreen mode.
- `--exit-btn` adds an Exit button.

Press `Esc` or `Q` to close the GUI. The run scripts include default text queries; run `build/clip_single` directly when you want a completely custom query list.

## Summary

This application combines a CPU text encoder and an asynchronous NPU image encoder in one C++ GUI. Text features are reusable, image features are produced continuously, and normalized dot products provide zero-shot text-image matching without a task-specific classifier.